# LED Classifier — Training Notebook (Google Colab)

**Stack:** TensorFlow/Keras CNN → TFLite INT8  
**Target:** ESP32-CAM (Freenove WROVER)  
**Classi:** red | green | blue | no_led

## Istruzioni per Colab:
1. File > Salva una copia in Drive
2. Runtime > Modifica tipo di runtime > GPU (T4)
3. Esegui le celle in ordine
4. Alla cella "Mount Drive" autorizza l'accesso

## Struttura attesa su Drive:
```
MyDrive/
└── led_classifier/
    └── dataset/
        └── augmented/
            ├── red/
            ├── green/
            ├── blue/
            └── no_led/
```

## Cella 1: Installazioni e Import

Installiamo le dipendenze richieste e importiamo tutte le librerie necessarie.

> In Colab esegui prima: `!pip install -q tensorflow==2.13.0`

In [ ]:
import os
import json
import shutil
import zipfile
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from pathlib import Path
from datetime import datetime
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             ConfusionMatrixDisplay)
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")
print(f"Data/ora   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Cella 2: Configurazione Centralizzata

Definiamo una classe di configurazione che centralizza tutti i parametri del progetto: percorsi, iperparametri, e impostazioni di training.

In [ ]:
class Config:
    # ── Percorsi ────────────────────────────────────────────────────────────
    # Salviamo il modello finito su Drive così non lo perdi
    DRIVE_ROOT    = Path("/content/drive/MyDrive/tiny_ml_led_classifier")
    OUTPUT_DIR    = DRIVE_ROOT / "output"
    MODEL_DIR     = OUTPUT_DIR / "models"
    LOG_DIR       = OUTPUT_DIR / "logs"
    PLOT_DIR      = OUTPUT_DIR / "plots"

    # MA LEGGIAMO IL DATASET DAL DISCO LOCALE VELOCE (Niente più errori Drive!)
    DATASET_DIR = Path('/content/dataset_veloce/augmented')

    # ── Dataset ─────────────────────────────────────────────────────────────
    CLASSES       = ["red", "green", "blue", "no_led"]
    IMG_SIZE      = (96, 96)
    IMG_SHAPE     = (96, 96, 3)
    NUM_CLASSES   = 4

    # ── Training ────────────────────────────────────────────────────────────
    BATCH_SIZE    = 32
    EPOCHS        = 60
    LEARNING_RATE = 1e-3
    VAL_SPLIT     = 0.15    # 15% validation
    TEST_SPLIT    = 0.10    # 10% test (held-out finale)
    SEED          = 42

    # ── Regularizzazione ────────────────────────────────────────────────────
    DROPOUT_RATE  = 0.4
    L2_LAMBDA     = 1e-4

    # ── Early stopping ──────────────────────────────────────────────────────
    ES_PATIENCE   = 12      # epoche senza miglioramento prima di fermarsi
    LR_PATIENCE   = 6       # epoche prima di dimezzare il LR

    # ── TFLite ──────────────────────────────────────────────────────────────
    CALIB_SAMPLES = 200     # frame dal training set per la calibrazione INT8

cfg = Config()

# Crea le directory di output
for d in [cfg.MODEL_DIR, cfg.LOG_DIR, cfg.PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Configurazione caricata ✓")
print(f"  Dataset   : {cfg.DATASET_DIR}")
print(f"  Output    : {cfg.OUTPUT_DIR}")

## Cella 3: Mount Google Drive

Montiamo Google Drive per salvare i modelli e verificare la disponibilità del dataset.

In [ ]:
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount("/content/drive")
else:
    print("Google Drive è già montato! Procedo...")

# Verifica che il dataset esista
for cls in cfg.CLASSES:
    cls_dir = cfg.DATASET_DIR / cls
    n = len(list(cls_dir.glob("*.png"))) if cls_dir.exists() else 0
    status = "✓" if n > 0 else "✗ MANCANTE"
    print(f"  {cls:<10} {n:>4} immagini  {status}")

## Cella 4: Costruzione Split Train/Val/Test

Dividiamo il dataset in tre sottoinsiemi (training, validation, test) in modo **stratificato per classe**. Questo garantisce che ogni classe sia ben rappresentata in ciascun split.

In [ ]:
import random
from collections import defaultdict

random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
tf.random.set_seed(cfg.SEED)


def build_splits(dataset_dir: Path,
                 classes: list[str],
                 val_split: float,
                 test_split: float,
                 seed: int = 42) -> tuple[list, list, list]:
    """
    Divide il dataset in train/val/test in modo stratificato per classe.
    Restituisce tre liste di tuple (filepath, label_index).
    """
    train_data, val_data, test_data = [], [], []

    for label_idx, cls in enumerate(classes):
        cls_dir = dataset_dir / cls
        files   = sorted(cls_dir.glob("*.png"))
        files   = [str(f) for f in files]
        random.Random(seed).shuffle(files)

        n       = len(files)
        n_test  = max(1, int(n * test_split))
        n_val   = max(1, int(n * val_split))
        n_train = n - n_test - n_val

        test_files  = files[:n_test]
        val_files   = files[n_test:n_test + n_val]
        train_files = files[n_test + n_val:]

        train_data += [(f, label_idx) for f in train_files]
        val_data   += [(f, label_idx) for f in val_files]
        test_data  += [(f, label_idx) for f in test_files]

        print(f"  {cls:<10}  tot={n:>4}  "
              f"train={len(train_files):>4}  "
              f"val={len(val_files):>3}  "
              f"test={len(test_files):>3}")

    return train_data, val_data, test_data


print("Split del dataset:")
train_data, val_data, test_data = build_splits(
    cfg.DATASET_DIR, cfg.CLASSES,
    cfg.VAL_SPLIT, cfg.TEST_SPLIT, cfg.SEED
)
print(f"\n  Totale train : {len(train_data)}")
print(f"  Totale val   : {len(val_data)}")
print(f"  Totale test  : {len(test_data)}")

## Cella 5: Dataset Pipeline (tf.data)

Costruiamo una pipeline TensorFlow ottimizzata per:
- **Carica e normalizzazione** delle immagini PNG
- **Augmentation on-the-fly** nel training set (flip, brightness, contrast)
- **Prefetching e caching** per massimizzare la velocità di I/O

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def parse_image(filepath: str, label: int):
    """Carica, decodifica e normalizza un'immagine."""
    img = tf.io.read_file(filepath)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, cfg.IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, tf.one_hot(label, cfg.NUM_CLASSES)


def make_dataset(data: list[tuple],
                 batch_size: int,
                 shuffle: bool = False,
                 augment: bool = False) -> tf.data.Dataset:
    """
    Costruisce una tf.data.Dataset ottimizzata (prefetch + cache).
    L'augmentation viene applicata solo al training set.
    """
    paths  = [d[0] for d in data]
    labels = [d[1] for d in data]

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(data), seed=cfg.SEED,
                        reshuffle_each_iteration=True)

    ds = ds.map(parse_image, num_parallel_calls=AUTOTUNE)

    if augment:
        ds = ds.map(tf_augment, num_parallel_calls=AUTOTUNE)

    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds


# Augmentation on-the-fly in TensorFlow (leggera, sul training set)
# Nota: non includiamo color jitter — stessa filosofia dello script offline
def tf_augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label


train_ds = make_dataset(train_data, cfg.BATCH_SIZE, shuffle=True,  augment=True)
val_ds   = make_dataset(val_data,   cfg.BATCH_SIZE, shuffle=False, augment=False)
test_ds  = make_dataset(test_data,  cfg.BATCH_SIZE, shuffle=False, augment=False)

print("Pipeline tf.data costruita ✓")

## Cella 6: Visualizzazione Campioni del Training Set

Visualizziamo alcuni campioni casuali dal training set per verificare che il caricamento e la normalizzazione funzionino correttamente.

In [ ]:
def show_samples(ds: tf.data.Dataset, classes: list[str], n: int = 16):
    imgs, labels = next(iter(ds))
    imgs   = imgs.numpy()[:n]
    labels = tf.argmax(labels, axis=1).numpy()[:n]

    cols = 8
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.7))
    axes = axes.flatten()

    CLASS_COLORS = {"red": "#e74c3c", "green": "#2ecc71",
                    "blue": "#3498db", "no_led": "#95a5a6"}

    for i, (img, lbl) in enumerate(zip(imgs, labels)):
        axes[i].imshow(img)
        cls_name = classes[lbl]
        axes[i].set_title(cls_name, fontsize=8,
                          color=CLASS_COLORS.get(cls_name, "white"),
                          fontweight="bold")
        axes[i].axis("off")

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Campioni Training Set (normalizzati 0-1)",
                 color="white", fontsize=11)
    fig.patch.set_facecolor("#1e1e2e")
    plt.tight_layout()
    plt.savefig(cfg.PLOT_DIR / "samples_preview.png", dpi=120,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"Preview salvata.")

show_samples(train_ds, cfg.CLASSES)

## Cella 7: Architettura CNN

Definiamo l'architettura della rete neurale convoluzionale.

### Design Rationale:
- **3 blocchi Conv+BN+ReLU+MaxPool** con profondità crescente (16→32→64 filtri)
- **GlobalAveragePooling** invece di Flatten: risparmia ~80% di parametri e aumenta robustezza
- **Dropout** prima del Dense: contrasta l'overfitting su dataset piccoli
- **L2 Regularization** sul Dense finale: regolarizzazione aggiuntiva
- **Softmax** in output: probabilità per 4 classi

### Stima risorse su ESP32 (dopo quantizzazione INT8):
- Flash: ~55-65 KB
- RAM peak: ~90-110 KB (con PSRAM abilitata: OK)
- Latenza: ~80-120 ms per inferenza

In [ ]:
def build_model(input_shape: tuple, num_classes: int) -> tf.keras.Model:

    inputs = layers.Input(shape=input_shape, name="input_96x96x3")

    # ── Blocco 1 ─────────────────────────────────────────────────
    x = layers.Conv2D(16, (3, 3), padding="same", use_bias=False,
                      name="conv1")(inputs)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.Activation("relu", name="relu1")(x)
    x = layers.MaxPooling2D((2, 2), name="pool1")(x)         # 48×48×16

    # ── Blocco 2 ─────────────────────────────────────────────────
    x = layers.Conv2D(32, (3, 3), padding="same", use_bias=False,
                      name="conv2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.Activation("relu", name="relu2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool2")(x)         # 24×24×32

    # ── Blocco 3 ─────────────────────────────────────────────────
    x = layers.Conv2D(64, (3, 3), padding="same", use_bias=False,
                      name="conv3")(x)
    x = layers.BatchNormalization(name="bn3")(x)
    x = layers.Activation("relu", name="relu3")(x)
    x = layers.MaxPooling2D((2, 2), name="pool3")(x)         # 12×12×64

    # ── Classificatore ───────────────────────────────────────────
    x = layers.GlobalAveragePooling2D(name="gap")(x)         # 64
    x = layers.Dropout(cfg.DROPOUT_RATE, name="dropout")(x)
    x = layers.Dense(
            num_classes,
            activation="softmax",
            kernel_regularizer=regularizers.l2(cfg.L2_LAMBDA),
            name="output"
        )(x)                                                  # 4

    model = models.Model(inputs, x, name="led_cnn")
    return model


model = build_model(cfg.IMG_SHAPE, cfg.NUM_CLASSES)
model.summary()

# Conta parametri trainable
total_params  = model.count_params()
print(f"\n  Parametri totali  : {total_params:,}")
print(f"  Stima flash INT8  : ~{total_params // 1024 + 12} KB")

## Cella 8: Compilazione e Callbacks

Compiliamo il modello e configuriamo i callback per:
- **ModelCheckpoint**: salvare il modello con la migliore `val_accuracy`
- **EarlyStopping**: fermare il training se non migliora per 12 epoche
- **ReduceLROnPlateau**: dimezzare il learning rate se `val_loss` non scende
- **CSVLogger**: registrare le metriche di training in un file CSV

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=cfg.LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy",
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

# ── Callbacks ────────────────────────────────────────────────────────────────

# Salva solo il modello con la miglior val_accuracy
best_model_path = str(cfg.MODEL_DIR / "led_cnn_best.h5")

cb_checkpoint = callbacks.ModelCheckpoint(
    filepath=best_model_path,
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

# Ferma il training se non migliora per ES_PATIENCE epoche
cb_early_stop = callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=cfg.ES_PATIENCE,
    restore_best_weights=True,
    verbose=1
)

# Dimezza il LR se val_loss non scende per LR_PATIENCE epoche
cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=cfg.LR_PATIENCE,
    min_lr=1e-6,
    verbose=1
)

# Log CSV per analisi post-training
cb_csv_logger = callbacks.CSVLogger(
    str(cfg.LOG_DIR / "training_log.csv"),
    separator=",",
    append=False
)

cb_list = [cb_checkpoint, cb_early_stop, cb_reduce_lr, cb_csv_logger]

print("Modello compilato ✓")
print(f"  Checkpoint → {best_model_path}")

## Cella 9: Training

Eseguiamo il training del modello. Il training si interrompe automaticamente se non migliora la validation accuracy per 12 epoche consecutive.

In [ ]:
print(f"\nAvvio training — max {cfg.EPOCHS} epoche")
print(f"  EarlyStopping patience : {cfg.ES_PATIENCE}")
print(f"  LR iniziale            : {cfg.LEARNING_RATE}\n")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=cfg.EPOCHS,
    callbacks=cb_list,
    verbose=1
)

print("\nTraining completato ✓")
print(f"  Epoche effettive   : {len(history.history['loss'])}")
print(f"  Miglior val_acc    : {max(history.history['val_accuracy']):.4f}")

## Cella 10: Grafici Training

Visualizziamo i grafici di Loss, Accuracy e Learning Rate durante il training.

In [ ]:
def plot_history(history, save_path: Path):
    hist    = history.history
    epochs  = range(1, len(hist["loss"]) + 1)
    best_ep = np.argmax(hist["val_accuracy"]) + 1

    fig = plt.figure(figsize=(16, 5), facecolor="#1e1e2e")
    gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

    PLOT_STYLE = dict(linewidth=2)
    GRID_STYLE = dict(color="#444466", linestyle="--", alpha=0.5)

    # ── Loss ─────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(epochs, hist["loss"],     color="#e74c3c", label="Train",  **PLOT_STYLE)
    ax1.plot(epochs, hist["val_loss"], color="#3498db", label="Val",    **PLOT_STYLE)
    ax1.axvline(best_ep, color="#f39c12", linestyle=":", linewidth=1.5,
                label=f"Best ep {best_ep}")
    ax1.set_title("Loss",       color="white", fontsize=12)
    ax1.set_xlabel("Epoca",     color="#aaaacc")
    ax1.set_ylabel("Cat. CE",   color="#aaaacc")
    ax1.legend(facecolor="#2a2a3e", labelcolor="white")
    ax1.grid(**GRID_STYLE)
    ax1.set_facecolor("#14141e")
    ax1.tick_params(colors="#aaaacc")

    # ── Accuracy ──────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1])
    ax2.plot(epochs, hist["accuracy"],     color="#e74c3c", label="Train", **PLOT_STYLE)
    ax2.plot(epochs, hist["val_accuracy"], color="#3498db", label="Val",   **PLOT_STYLE)
    ax2.axvline(best_ep, color="#f39c12", linestyle=":", linewidth=1.5)
    best_val = max(hist["val_accuracy"])
    ax2.annotate(f"{best_val:.3f}",
                 xy=(best_ep, best_val),
                 xytext=(best_ep + 1, best_val - 0.05),
                 color="#f39c12", fontsize=9,
                 arrowprops=dict(arrowstyle="->", color="#f39c12"))
    ax2.set_title("Accuracy",  color="white", fontsize=12)
    ax2.set_xlabel("Epoca",    color="#aaaacc")
    ax2.set_ylim(0, 1.05)
    ax2.legend(facecolor="#2a2a3e", labelcolor="white")
    ax2.grid(**GRID_STYLE)
    ax2.set_facecolor("#14141e")
    ax2.tick_params(colors="#aaaacc")

    # ── Learning Rate ─────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[2])
    lr_vals = hist.get("lr", [cfg.LEARNING_RATE] * len(epochs))
    ax3.semilogy(epochs, lr_vals, color="#2ecc71", **PLOT_STYLE)
    ax3.set_title("Learning Rate (log)",  color="white", fontsize=12)
    ax3.set_xlabel("Epoca",               color="#aaaacc")
    ax3.set_ylabel("LR",                  color="#aaaacc")
    ax3.grid(**GRID_STYLE)
    ax3.set_facecolor("#14141e")
    ax3.tick_params(colors="#aaaacc")

    plt.suptitle("LED Classifier — Training History",
                 color="white", fontsize=14, y=1.02)
    plt.savefig(save_path, dpi=140, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()
    print(f"Grafico salvato: {save_path.name}")

plot_history(history, cfg.PLOT_DIR / "training_history.png")

## Cella 11: Valutazione sul Test Set (held-out)

Valutiamo il modello sul test set e creiamo una matrice di confusione e un report di classificazione.

In [ ]:
print("Valutazione sul test set (held-out)...")

# Carica il best checkpoint
model.load_weights(best_model_path)
test_loss, test_acc, test_prec, test_rec = model.evaluate(test_ds, verbose=0)

print(f"\n  Test Loss      : {test_loss:.4f}")
print(f"  Test Accuracy  : {test_acc:.4f}  ({test_acc*100:.1f}%)")
print(f"  Test Precision : {test_prec:.4f}")
print(f"  Test Recall    : {test_rec:.4f}")

# ── Matrice di Confusione ────────────────────────────────────────────────────
all_true, all_pred = [], []
for imgs_batch, labels_batch in test_ds:
    preds  = model.predict(imgs_batch, verbose=0)
    all_true.extend(tf.argmax(labels_batch, axis=1).numpy())
    all_pred.extend(np.argmax(preds, axis=1))

all_true = np.array(all_true)
all_pred = np.array(all_pred)

cm = confusion_matrix(all_true, all_pred)

fig, ax = plt.subplots(figsize=(7, 6), facecolor="#1e1e2e")
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=cfg.CLASSES
)
disp.plot(
    ax=ax,
    colorbar=True,
    cmap="Blues",
    values_format="d"
)
ax.set_title("Matrice di Confusione — Test Set",
             color="white", fontsize=13, pad=14)
ax.set_facecolor("#14141e")
ax.tick_params(colors="#ccccee")
ax.xaxis.label.set_color("#ccccee")
ax.yaxis.label.set_color("#ccccee")
plt.setp(ax.get_xticklabels(), color="#ccccee")
plt.setp(ax.get_yticklabels(), color="#ccccee")
fig.patch.set_facecolor("#1e1e2e")
plt.tight_layout()
plt.savefig(cfg.PLOT_DIR / "confusion_matrix.png", dpi=140,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

# ── Rapporto di Classificazione ──────────────────────────────────────────────
report = classification_report(all_true, all_pred,
                                target_names=cfg.CLASSES,
                                digits=3)
print("\nRapporto di Classificazione:")
print(report)

with open(cfg.LOG_DIR / "classification_report.txt", "w") as f:
    f.write(report)

## Cella 12: Conversione TFLite FLOAT32 (baseline)

Convertiamo il modello Keras in formato TFLite a precisione FLOAT32. Questo serve come baseline per il confronto con il modello quantizzato.

In [ ]:
print("Conversione TFLite FLOAT32...")

converter_f32      = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_f32_model   = converter_f32.convert()
tflite_f32_path    = cfg.MODEL_DIR / "led_classifier_f32.tflite"

with open(tflite_f32_path, "wb") as f:
    f.write(tflite_f32_model)

print(f"  Dimensione FLOAT32 : {len(tflite_f32_model) / 1024:.1f} KB")
print(f"  Salvato            : {tflite_f32_path.name}")

## Cella 13: Conversione TFLite INT8 (full integer quantization)

Convertiamo il modello con **quantizzazione INT8 completa**:
- Tutti i pesi e le attivazioni vengono quantizzati a INT8
- Richiede un "representative dataset" per calibrare i range
- Risultato: modello ~4x più piccolo, ~2-4x più veloce su MCU
- Input/Output: FLOAT32 (più semplice da gestire sul firmware)

In [ ]:
def representative_dataset_gen():
    """
    Fornisce al converter un sottoinsieme del training set per calibrazione.
    TensorFlow usa questi campioni per calcolare i range min/max delle attivazioni.
    """
    count = 0
    for imgs_batch, _ in train_ds:
        for img in imgs_batch:
            if count >= cfg.CALIB_SAMPLES:
                return
            yield [tf.expand_dims(img, axis=0)]
            count += 1

print(f"Conversione TFLite INT8 (calibrazione su {cfg.CALIB_SAMPLES} campioni)...")

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)

# Attiva la quantizzazione ottimizzata di default
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]

# Fornisce il dataset di calibrazione
converter_int8.representative_dataset = representative_dataset_gen

# Forza tutti gli operatori a INT8
converter_int8.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

# I/O rimangono FLOAT32 (l'ESP32 gestisce la conversione internamente)
converter_int8.inference_input_type  = tf.float32
converter_int8.inference_output_type = tf.float32

tflite_int8_model = converter_int8.convert()
tflite_int8_path  = cfg.MODEL_DIR / "led_classifier_int8.tflite"

with open(tflite_int8_path, "wb") as f:
    f.write(tflite_int8_model)

size_f32  = len(tflite_f32_model)  / 1024
size_int8 = len(tflite_int8_model) / 1024
ratio     = size_f32 / size_int8

print(f"\n  FLOAT32 : {size_f32:.1f} KB")
print(f"  INT8    : {size_int8:.1f} KB")
print(f"  Ratio   : {ratio:.2f}x compressione")
print(f"  Salvato : {tflite_int8_path.name}")

## Cella 14: Verifica Accuratezza TFLite INT8 vs Keras

È fondamentale verificare che la quantizzazione non abbia degradato l'accuratezza in modo significativo. Una perdita < 2% è considerata accettabile.

In [ ]:
def eval_tflite(tflite_model_bytes: bytes,
                test_data: list,
                classes: list[str]) -> float:
    """Esegue l'inferenza con l'interpreter TFLite e calcola l'accuracy."""
    interpreter = tf.lite.Interpreter(model_content=tflite_model_bytes)
    interpreter.allocate_tensors()

    in_idx  = interpreter.get_input_details()[0]["index"]
    out_idx = interpreter.get_output_details()[0]["index"]

    correct = 0
    for filepath, true_label in test_data:
        img = tf.io.read_file(filepath)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, cfg.IMG_SIZE)
        img = tf.cast(img, tf.float32) / 255.0
        img = tf.expand_dims(img, axis=0).numpy()

        interpreter.set_tensor(in_idx, img)
        interpreter.invoke()
        pred_label = np.argmax(interpreter.get_tensor(out_idx))

        if pred_label == true_label:
            correct += 1

    return correct / len(test_data)


print("Confronto accuratezze sul test set:")
keras_acc  = test_acc
f32_acc    = eval_tflite(tflite_f32_model,  test_data, cfg.CLASSES)
int8_acc   = eval_tflite(tflite_int8_model, test_data, cfg.CLASSES)

print(f"\n  Keras  (float32) : {keras_acc:.4f}  ({keras_acc*100:.1f}%)")
print(f"  TFLite (float32) : {f32_acc:.4f}  ({f32_acc*100:.1f}%)")
print(f"  TFLite (int8)    : {int8_acc:.4f}  ({int8_acc*100:.1f}%)")
print(f"\n  Degradazione INT8 vs Keras: {(keras_acc - int8_acc)*100:.2f}%")

if abs(keras_acc - int8_acc) < 0.02:
    print("  ✅ Degradazione accettabile (<2%) — modello pronto per il deploy")
else:
    print("  ⚠️  Degradazione >2% — considera di aumentare CALIB_SAMPLES")

## Cella 15: Genera il File .h per ESP32 (array C)

TFLite Micro su ESP32 richiede il modello come array C statico da includere nel firmware come header file.

In [ ]:
def tflite_to_c_header(tflite_bytes: bytes,
                        array_name: str,
                        output_path: Path) -> None:
    """
    Converte il file .tflite in un header C contenente il modello
    come array di uint8_t — pronto per #include nel firmware ESP32.
    """
    n        = len(tflite_bytes)
    hex_vals = ", ".join(f"0x{b:02x}" for b in tflite_bytes)
    chunks   = [hex_vals[i:i+80] for i in range(0, len(hex_vals), 80)]
    hex_body = "\n    ".join(chunks)

    header = f"""\
// Auto-generato da train.py — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
// Modello: LED Classifier CNN (quantizzato INT8)
// Classi : red=0, green=1, blue=2, no_led=3
// Input  : float32[1][96][96][3]  (normalizzato 0.0-1.0)
// Output : float32[1][4]          (probabilità softmax)
// Size   : {n} bytes  ({n/1024:.1f} KB)

#pragma once
#include <stdint.h>

alignas(8) const uint8_t {array_name}[] = {{
    {hex_body}
}};

const unsigned int {array_name}_len = {n};
"""
    with open(output_path, "w") as f:
        f.write(header)

    print(f"  Header C generato : {output_path.name}")
    print(f"  Array name        : {array_name}")
    print(f"  Size              : {n} bytes ({n/1024:.1f} KB)")


header_path = cfg.MODEL_DIR / "led_model_data.h"
tflite_to_c_header(tflite_int8_model, "led_model_data", header_path)

## Cella 16: Salvataggio Metadati e Riepilogo Finale

Salviamo tutti i metadati del training e del modello in un file JSON per documentazione e tracking.

In [ ]:
metadata = {
    "generated_at"        : datetime.now().isoformat(),
    "model_name"          : "led_cnn",
    "classes"             : cfg.CLASSES,
    "input_shape"         : list(cfg.IMG_SHAPE),
    "input_dtype"         : "float32",
    "input_normalization" : "divide by 255",
    "output_shape"        : [cfg.NUM_CLASSES],
    "output_dtype"        : "float32",
    "training": {
        "epochs_run"      : len(history.history["loss"]),
        "best_val_acc"    : float(max(history.history["val_accuracy"])),
        "final_train_acc" : float(history.history["accuracy"][-1]),
        "optimizer"       : "Adam",
        "learning_rate"   : cfg.LEARNING_RATE,
        "batch_size"      : cfg.BATCH_SIZE,
    },
    "evaluation": {
        "test_accuracy_keras"  : float(keras_acc),
        "test_accuracy_f32"    : float(f32_acc),
        "test_accuracy_int8"   : float(int8_acc),
        "int8_degradation_pct" : float((keras_acc - int8_acc) * 100),
    },
    "tflite": {
        "f32_size_kb"  : round(size_f32,  2),
        "int8_size_kb" : round(size_int8, 2),
        "compression"  : round(ratio, 2),
    },
    "artifacts": {
        "keras_model"   : "models/led_cnn_best.h5",
        "tflite_f32"    : "models/led_classifier_f32.tflite",
        "tflite_int8"   : "models/led_classifier_int8.tflite",
        "c_header"      : "models/led_model_data.h",
    }
}

with open(cfg.OUTPUT_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\n" + "=" * 60)
print(" RIEPILOGO FINALE")
print("=" * 60)
print(f"  Test accuracy (INT8) : {int8_acc*100:.1f}%")
print(f"  Dimensione modello   : {size_int8:.1f} KB")
print(f"  Artefatti generati:")
print(f"    ├─ led_cnn_best.h5")
print(f"    ├─ led_classifier_f32.tflite")
print(f"    ├─ led_classifier_int8.tflite")
print(f"    └─ led_model_data.h      ← copia questo nel firmware ESP32")
print("=" * 60)
print("\n✅ Prossimo step: integra led_model_data.h nel progetto PlatformIO")